# packet_analyze_v2 — LLM 판정 (Colab T4 + Ollama)

로컬에서 `analyze.sh`로 만든 `report/*.evidence.json`을 LLM에 주고 판정(verdict)을 받는다.

**전제**: 런타임 → 유형 변경 → **T4 GPU** 선택.

흐름: repo clone → Ollama 설치 → qwen2.5:14b 다운로드 → `llm_analyze.py` 실행

모델: qwen2.5:14b (Q4_K_M, ~9GB) — T4 16GB VRAM에 여유롭게 들어감. function-calling 지원.

In [ ]:
# 1. repo clone
REPO = "https://github.com/DAADAISMYLIFE/packet_analyze_v2.git"
!git clone -q $REPO /content/packet_analyze_v2
%cd /content/packet_analyze_v2
!ls report/*.evidence.json

In [ ]:
# 2. Ollama 설치
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 3. vLLM OpenAI 서버 백그라운드 부팅
#    Qwen2.5-14B-Instruct-AWQ: T4(16GB)에서 ~9GB 사용, function-calling 정확도 7B 대비 우수
#    --tool-call-parser hermes : Qwen2.5 function-calling 파서
import subprocess, os
os.makedirs('/content/logs', exist_ok=True)
srv = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', 'Qwen/Qwen2.5-14B-Instruct-AWQ',
    '--max-model-len', '16384',
    '--gpu-memory-utilization', '0.90',
    '--enable-auto-tool-choice',
    '--tool-call-parser', 'hermes',
], stdout=open('/content/logs/vllm.log','w'), stderr=subprocess.STDOUT)
print('vLLM 부팅 중... (모델 다운로드 포함 5~8분)')

In [ ]:
# 3. Ollama 서버 백그라운드 시작 + 모델 다운로드
import subprocess, os, time

os.makedirs('/content/logs', exist_ok=True)
subprocess.Popen(
    ['ollama', 'serve'],
    stdout=open('/content/logs/ollama.log', 'w'),
    stderr=subprocess.STDOUT,
)
time.sleep(3)  # 서버 초기화 대기

print('모델 다운로드 중... (qwen2.5:14b ~9GB, 몇 분 걸림)')
subprocess.run(['ollama', 'pull', 'qwen2.5:14b'], check=True)
print('✅ 완료')

In [ ]:
# 4. 서버 준비 확인
import urllib.request, time
for i in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434', timeout=2)
        print('✅ Ollama 준비 완료'); break
    except Exception:
        time.sleep(3)
        if i % 5 == 0: print(f'  대기 {i*3}s...')
else:
    print('❌ 타임아웃 — /content/logs/ollama.log 확인')
    !tail -20 /content/logs/ollama.log

In [ ]:
# 5. LLM 판정 — report/ 의 모든 evidence.json을 순서대로 처리
import glob, os

MODEL = 'qwen2.5:14b'
BASE_URL = 'http://localhost:11434/v1'

evidence_files = sorted(glob.glob('report/*.evidence.json'))
print(f'evidence {len(evidence_files)}개 발견:')
for f in evidence_files:
    print(f'  {f}')
print()

for ev_path in evidence_files:
    name = os.path.basename(ev_path).replace('.evidence.json', '')
    print(f'\n{"="*60}')
    print(f'  분석: {name}')
    print(f'{"="*60}')
    !python scripts/llm_analyze.py {name} \
        --base-url {BASE_URL} \
        --model {MODEL} \
        --temperature 0.0